In [ ]:
import os, csv, json, glob, cv2, requests, zipfile
import numpy as np
from tqdm import tqdm
import mediapipe as mp
from pathlib import Path

# ---------- CONFIG ----------
DATA_ROOT = Path("FYPB_How2Sign")
CLIPS_DIR = DATA_ROOT / "clips"
SEQ_DIR = DATA_ROOT / "sequences"
ANN_FILE = DATA_ROOT / "annotations.csv"
SUBSET_SIZE = 100  # how many videos to fetch for a quick start

# ---------- DOWNLOAD ----------
def download_subset():
    DATA_ROOT.mkdir(exist_ok=True)
    print("Downloading How2Sign subset metadata...")
    url = "https://raw.githubusercontent.com/how2sign/how2sign-dataset/main/how2sign_train.json"
    r = requests.get(url)
    r.raise_for_status()
    meta = r.json()
    subset = meta[:SUBSET_SIZE]
    CLIPS_DIR.mkdir(exist_ok=True)
    ann_rows = []
    for i, item in enumerate(tqdm(subset)):
        vid_url = item["url"]
        sent = item.get("text", "")
        fname = f"clip_{i:04d}.mp4"
        out_path = CLIPS_DIR / fname
        if not out_path.exists():
            try:
                v = requests.get(vid_url, stream=True, timeout=20)
                with open(out_path, "wb") as f:
                    for chunk in v.iter_content(chunk_size=8192):
                        f.write(chunk)
            except Exception as e:
                print("Skip", vid_url, e)
                continue
        ann_rows.append([fname, sent])
    with open(ANN_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(ann_rows)
    print("Saved annotations.csv with", len(ann_rows), "rows.")

# ---------- LANDMARK EXTRACTION ----------
mp_holistic = mp.solutions.holistic

def extract_landmarks(video_path, seq_len=64, fps_target=25):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    orig_fps = cap.get(cv2.CAP_PROP_FPS) or fps_target
    step = max(1, int(round(orig_fps / fps_target)))
    frames = []
    with mp_holistic.Holistic(model_complexity=1) as holistic:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            if idx % step != 0:
                idx += 1; continue
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = holistic.process(img)
            feat = []
            for lm_list in [res.left_hand_landmarks, res.right_hand_landmarks]:
                if lm_list:
                    for lm in lm_list.landmark:
                        feat.extend([lm.x, lm.y, lm.z])
                else:
                    feat.extend([0.0]*21*3)
            if res.pose_landmarks:
                for lm in res.pose_landmarks.landmark:
                    feat.extend([lm.x, lm.y, lm.z])
            else:
                feat.extend([0.0]*33*3)
            frames.append(feat)
            idx += 1
    cap.release()
    arr = np.array(frames)
    if arr.shape[0]==0: return None
    if arr.shape[0] < seq_len:
        pad = np.zeros((seq_len-arr.shape[0], arr.shape[1]))
        arr = np.vstack([arr,pad])
    else:
        arr = arr[:seq_len]
    return arr.astype(np.float32)

def build_sequences():
    SEQ_DIR.mkdir(exist_ok=True)
    with open(ANN_FILE, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        entries = list(reader)
    for fname, sent in tqdm(entries):
        vid_path = CLIPS_DIR / fname
        out_path = SEQ_DIR / (Path(fname).stem + ".npz")
        if out_path.exists(): continue
        arr = extract_landmarks(vid_path)
        if arr is None: continue
        np.savez_compressed(out_path, arr=arr, sentence=sent)
    print("Sequences saved to", SEQ_DIR)

if __name__ == "__main__":
    download_subset()
    build_sequences()
    print("✅ How2Sign subset ready.")

Starting downloader + adapter for preprocessed How2Sign (SignDiff).
[download] Attempting: https://huggingface.co/datasets/SignDiff/how2sign-pre/resolve/main/how2sign-pre.zip (try 1)
  download failed: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/SignDiff/how2sign-pre/resolve/main/how2sign-pre.zip
[download] Attempting: https://huggingface.co/datasets/SignDiff/how2sign-pre/resolve/main/how2sign-pre.zip (try 2)
  download failed: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/SignDiff/how2sign-pre/resolve/main/how2sign-pre.zip
[download] Attempting: https://huggingface.co/datasets/SignDiff/how2sign-pre/resolve/main/how2sign-pre.zip (try 3)
  download failed: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/SignDiff/how2sign-pre/resolve/main/how2sign-pre.zip
  moving to next URL if available...
Download failed: All download attempts failed. Last exception: HTTPError('401 Client Error: Unauthorized for url: https:/